# Notebook 04 — Model Evaluation

**Project**: Diabetes Prediction  
**Goal**: Comprehensive evaluation of all trained models.

Steps:
1. Imports and load models + test data
2. Calculate all metrics for each model
3. Confusion matrices
4. ROC curves overlay
5. Precision-Recall curves
6. Feature importance plots
7. Model comparison bar charts
8. Learning curves
9. Overfitting check
10. Best model selection

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import (
    FEATURE_COLUMNS, TARGET_COLUMN, RANDOM_STATE, MODELS_DIR,
    PROCESSED_DATA_DIR, PLOTS_DIR, REPORTS_DIR, PERFORMANCE_TARGETS
)
from src.models import load_model
from src.evaluation import (
    ModelEvaluator, calculate_metrics, per_class_metrics,
    confusion_matrix_metrics, classification_report_df,
    compare_models, check_overfitting
)
from src.visualization import (
    plot_confusion_matrix, plot_roc_curves, plot_precision_recall_curves,
    plot_feature_importance, plot_model_comparison, plot_learning_curve
)

print('Imports successful.')

## 1. Load Models and Test Data

In [ ]:
def load_data_and_models():
    """Load processed test data and all saved models."""
    x_test_path = PROCESSED_DATA_DIR / 'X_test_scaled.npy'
    y_test_path = PROCESSED_DATA_DIR / 'y_test.csv'
    x_train_path = PROCESSED_DATA_DIR / 'X_train_scaled.npy'
    y_train_path = PROCESSED_DATA_DIR / 'y_train_resampled.csv'

    if x_test_path.exists():
        X_test  = np.load(str(x_test_path))
        y_test  = pd.read_csv(y_test_path).squeeze()
        X_train = np.load(str(x_train_path))
        y_train = pd.read_csv(y_train_path).squeeze()
        print('Data loaded from processed files.')
    else:
        print('Processed files not found — generating synthetic data...')
        from sklearn.datasets import make_classification
        from sklearn.model_selection import train_test_split
        from sklearn.preprocessing import StandardScaler
        X, y = make_classification(
            n_samples=3000, n_features=21, n_informative=15,
            n_classes=2, n_clusters_per_class=1, random_state=RANDOM_STATE
        )
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
        )
        y_train = pd.Series(y_train, name=TARGET_COLUMN)
        y_test  = pd.Series(y_test,  name=TARGET_COLUMN)

    # Load saved models
    models = {}
    model_names = ['LogisticRegression', 'RandomForest', 'XGBoost', 'LightGBM']
    for name in model_names:
        path = MODELS_DIR / f'{name}.pkl'
        if path.exists():
            models[name] = load_model(str(path))
            print(f'  Loaded: {name}')
        else:
            print(f'  Not found, training fresh: {name}')
            from src.models import get_logistic_regression, get_random_forest, get_xgboost, get_lightgbm, train_model
            factory = {
                'LogisticRegression': lambda: get_logistic_regression(),
                'RandomForest': lambda: get_random_forest({'n_estimators': 50}),
                'XGBoost': lambda: get_xgboost({'n_estimators': 50}),
                'LightGBM': lambda: get_lightgbm({'n_estimators': 50}),
            }
            m = factory[name]()
            train_model(m, X_train, y_train)
            models[name] = m

    return X_train, X_test, y_train, y_test, models


X_train, X_test, y_train, y_test, models = load_data_and_models()
print(f'\nTest set: {X_test.shape}, models: {list(models.keys())}')

## 2. Calculate All Metrics

In [ ]:
evaluator = ModelEvaluator()
predictions = {}  # store predictions and probs for visualisations

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)
    y_pred_train = model.predict(X_train)
    y_prob_train = model.predict_proba(X_train)

    evaluator.evaluate(name, y_test, y_pred, y_prob)
    evaluator.evaluate_train(name, y_train, y_pred_train, y_prob_train)

    predictions[name] = {
        'y_true': y_test.values,
        'y_pred': y_pred,
        'y_prob': y_prob,
    }

print('\nMetrics calculation complete.')
evaluator.get_report()

In [ ]:
# Per-class metrics for the best model (by ROC-AUC)
best_name = evaluator.best_model(metric='roc_auc')
print(f'Best model by ROC-AUC: {best_name}')
print()
per_class = per_class_metrics(predictions[best_name]['y_true'], predictions[best_name]['y_pred'])
print(f'Per-class metrics for {best_name}:')
print(per_class.round(4))

In [ ]:
# Classification report for best model
report_df = classification_report_df(
    predictions[best_name]['y_true'],
    predictions[best_name]['y_pred']
)
print(f'Classification report — {best_name}:')
report_df

## 3. Confusion Matrices

In [ ]:
for name in models:
    plot_confusion_matrix(
        predictions[name]['y_true'],
        predictions[name]['y_pred'],
        model_name=name,
        save_path=str(PLOTS_DIR / f'confusion_matrix_{name}.png')
    )

## 4. ROC Curves Overlay

In [ ]:
plot_roc_curves(
    predictions,
    save_path=str(PLOTS_DIR / 'roc_curves.png')
)

## 5. Precision-Recall Curves

In [ ]:
plot_precision_recall_curves(
    predictions,
    save_path=str(PLOTS_DIR / 'precision_recall_curves.png')
)

## 6. Feature Importance Plots

In [ ]:
feature_names = FEATURE_COLUMNS

for name in ['RandomForest', 'XGBoost', 'LightGBM']:
    model = models.get(name)
    if model is None:
        continue
    importance = model.feature_importances_
    imp_df = pd.DataFrame({'feature': feature_names, 'importance': importance})
    plot_feature_importance(
        imp_df, top_n=15,
        save_path=str(PLOTS_DIR / f'feature_importance_{name}.png')
    )
    print(f'Feature importance plotted for {name}.')

## 7. Model Comparison Bar Charts

In [ ]:
comparison_df = evaluator.get_report()
metrics_to_plot = ['accuracy', 'roc_auc', 'f1_weighted', 'f1_macro']

plot_model_comparison(
    comparison_df,
    metrics=metrics_to_plot,
    save_path=str(PLOTS_DIR / 'model_comparison.png')
)

print('Comparison DataFrame:')
comparison_df

## 8. Learning Curves

In [ ]:
# Learning curve for the best model
best_model = models[best_name]
print(f'Plotting learning curve for {best_name}...')
plot_learning_curve(
    best_model, X_train, y_train, cv=3,
    save_path=str(PLOTS_DIR / f'learning_curve_{best_name}.png')
)

## 9. Overfitting Check

In [ ]:
print('=== OVERFITTING ANALYSIS ===')
for name in models:
    y_pred = predictions[name]['y_pred']
    y_prob = predictions[name]['y_prob']
    y_pred_train = models[name].predict(X_train)
    y_prob_train = models[name].predict_proba(X_train)

    test_metrics  = calculate_metrics(y_test,  y_pred,       y_prob)
    train_metrics = calculate_metrics(y_train, y_pred_train, y_prob_train)

    overfit_info = check_overfitting(train_metrics, test_metrics, threshold=0.05)
    overall = overfit_info.pop('overall_overfit', False)
    status = 'OVERFIT' if overall else 'OK'

    print(f'\n{name} [{status}]')
    for metric, vals in overfit_info.items():
        flag = '***' if vals['overfit'] else ''
        print(f'  {metric:<25}: train={vals["train"]:.4f}  test={vals["test"]:.4f}  gap={vals["gap"]:.4f} {flag}')

## 10. Best Model Selection

### Selection Criteria

We evaluate models against the following performance targets:
- Accuracy ≥ 0.70
- ROC-AUC ≥ 0.75
- F1-weighted ≥ 0.65

In [ ]:
report = evaluator.get_report()

print('=== PERFORMANCE TARGETS CHECK ===')
for name in report.index:
    row = report.loc[name]
    passes = []
    fails  = []
    for metric, threshold in PERFORMANCE_TARGETS.items():
        if metric in row.index:
            val = float(row[metric])
            if val >= threshold:
                passes.append(f'{metric}={val:.4f} >= {threshold}')
            else:
                fails.append(f'{metric}={val:.4f} < {threshold}')
    status = 'PASS' if not fails else 'PARTIAL'
    print(f'\n{name} [{status}]')
    for p in passes:
        print(f'  PASS: {p}')
    for f in fails:
        print(f'  FAIL: {f}')

In [ ]:
best = evaluator.best_model(metric='roc_auc')
best_metrics = report.loc[best]

print(f'\n=== BEST MODEL: {best} ===')
print(best_metrics.to_string())

# Save the best model name
with open(str(REPORTS_DIR / 'best_model.txt'), 'w') as f:
    f.write(best)

# Save full report
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
report.to_csv(str(REPORTS_DIR / 'evaluation_report.csv'))
print(f'\nEvaluation report saved to {REPORTS_DIR / "evaluation_report.csv"}')

### Justification for Best Model Selection

The best model is selected based on **ROC-AUC** (macro OVR), which is the most robust metric for imbalanced multi-class classification because:

1. It evaluates the model's ability to distinguish between all three classes.
2. It is insensitive to class imbalance (unlike accuracy).
3. It captures the tradeoff between sensitivity and specificity.

**LightGBM and XGBoost** typically outperform Logistic Regression and Random Forest on tabular health data due to:
- Gradient boosting captures non-linear feature interactions.
- Built-in regularisation reduces overfitting.
- Faster training than Random Forest with comparable or better accuracy.

Next step: `05_final_predictions.ipynb`